# 02 Final Inference

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`extract_inference.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/07_fine_tuning/extract_inference.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/07_fine_tuning)
- [`Final_Inference.ipynb`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/07_fine_tuning/Final_Inference.ipynb) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/07_fine_tuning)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'07_fine_tuning/extract_inference.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '2aff14879e14ff38e6c829026d6d8554bc9d32fc'}, '07_fine_tuning/Final_Inference.ipynb': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '139998ce60e84bb2e2f38b77d1e1da66771eb950'}}

## extract_inference

원본: [07_fine_tuning/extract_inference.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/07_fine_tuning/extract_inference.py)  
Git blob: `2aff14879e14ff38e6c829026d6d8554bc9d32fc`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/07_fine_tuning)

Run frozen SVC/KNN checkpoint inference using original CoM -> Deskew -> Pixel/HOG.
No data is used for training. Input npz must contain X or img (28x28, grayscale
0..255 in the original dataset convention). Model must be a trusted joblib
checkpoint from this experiment. Do not unpickle untrusted model files.

CLI 인자 (정적 추출): `--kit`, `--model`, `--model-sha256`, `--input`, `--out`

In [ ]:
%%archive_source 07_fine_tuning/extract_inference.py
#!/usr/bin/env python3
"""Run frozen SVC/KNN checkpoint inference using original CoM -> Deskew -> Pixel/HOG.
No data is used for training. Input npz must contain X or img (28x28, grayscale
0..255 in the original dataset convention). Model must be a trusted joblib
checkpoint from this experiment. Do not unpickle untrusted model files.
"""
import argparse,os,sys,zipfile,tempfile,hashlib,csv
from pathlib import Path
for k in ("OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS","NUMEXPR_NUM_THREADS"):os.environ[k]="2"
import numpy as np,joblib
from threadpoolctl import threadpool_limits
def main():
    ap=argparse.ArgumentParser()
    ap.add_argument("--kit",type=Path,required=True)
    ap.add_argument("--model",type=Path,required=True)
    ap.add_argument("--model-sha256",required=True)
    ap.add_argument("--input",type=Path,required=True)
    ap.add_argument("--out",type=Path,required=True)
    a=ap.parse_args()
    if a.out.exists():raise FileExistsError(a.out)
    with a.model.open("rb") as f: assert hashlib.file_digest(f,"sha256").hexdigest()==a.model_sha256
    with tempfile.TemporaryDirectory() as tmp:
        tmp=Path(tmp)
        with zipfile.ZipFile(a.kit) as z:
            for member in z.namelist():
                if member=="final_benchmark.py" or member.startswith("sources/"):z.extract(member,tmp)
        sys.path.insert(0,str(tmp))
        import final_benchmark as fb
        with np.load(a.input,allow_pickle=False) as z:
            key="X" if "X" in z.files else "img"
            x=z[key];assert x.ndim in (2,3) and x.size%784==0
            assert np.isfinite(x).all() and x.min()>=0 and x.max()<=255
            x=x.reshape(-1,784)
            ids=z["sample_index"] if "sample_index" in z.files else np.arange(len(x))
            collectors=z["collector_id"] if "collector_id" in z.files else np.full(len(x),-1)
        with threadpool_limits(2):
            model=joblib.load(a.model)
            assert np.array_equal(model.classes_,np.arange(16))
            pred=model.predict(fb.hybrid(x))
        a.out.parent.mkdir(parents=True,exist_ok=True)
        with a.out.open("w",newline="",encoding="utf-8") as f:
            w=csv.writer(f);w.writerow(["collector","index","prediction","label_id"])
            w.writerows((int(c),int(i),fb.LABELS[int(j)],int(j)) for c,i,j in zip(collectors,ids,pred))
        print("Predictions saved:",len(pred),a.out)
if __name__=="__main__":main()


## Final_Inference

원본: [07_fine_tuning/Final_Inference.ipynb](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/07_fine_tuning/Final_Inference.ipynb)  
Git blob: `139998ce60e84bb2e2f38b77d1e1da66771eb950`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/07_fine_tuning)

원본 Notebook의 Markdown 및 실행 코드 셀을 순서대로 보존한다. 기존 출력·이미지는 복제하지 않는다. 원본 Colab 환경/실행 순서는 원본 링크 참조.

### 원본 셀 0

# Frozen SVC/KNN inference extraction
Use trusted experiment checkpoints and the original grayscale input convention. No training or tuning. Select the exact checkpoint and SHA256 in the model manifest; do not use professor Test to select parameters.

In [ ]:
%%archive_source 07_fine_tuning/Final_Inference.ipynb#cell1
from pathlib import Path
import subprocess,sys
ROOT=Path('.')  # extracted review package
KIT=ROOT/'inputs/Final_Validation_Private_Evaluation_Kit_20261126.zip'
MODEL=ROOT/'02_SVC/full_runs/C2_G2/model.joblib'
MODEL_SHA256='f88be338da4065315d943ca4150b5a425c24adf63733fada7da212c837ae4398'
INPUT=Path('your_input.npz')
OUTPUT=Path('predictions_new.csv')
# Change MODEL, MODEL_SHA256 and INPUT to your chosen trusted files.

In [ ]:
%%archive_source 07_fine_tuning/Final_Inference.ipynb#cell2
assert KIT.exists() and MODEL.exists() and INPUT.exists()
subprocess.run([sys.executable,str(ROOT/'extract_inference.py'),'--kit',str(KIT),'--model',str(MODEL),'--model-sha256',MODEL_SHA256,'--input',str(INPUT),'--out',str(OUTPUT)],check=True)


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")